In [33]:
import os
import json
import time
import random
from datetime import datetime

In [34]:
# Configuración de rutas (apuntando a la carpeta montada en el contenedor)
BRONZE_PATH = "/home/jovyan/work/lakehouse/bronze/"# Configuración de los sensores de la fábrica ficticia
SENSORES_CONFIG = {
    "SENSOR_01": {"temp_base": 50.0, "volt_base": 220.0, "vib_base": 0.05},
    "SENSOR_02": {"temp_base": 80.0, "volt_base": 380.0, "vib_base": 0.12},
    "SENSOR_03": {"temp_base": 50.0, "volt_base": 220.0, "vib_base": 0.02}
}

In [ ]:
def generar_lectura_sensor(sensor_id, config):
    """Genera una lectura de sensor con posibilidad de ruido, nulos o anomalías"""
    probabilidad = random.random()
    timestamp = datetime.now().isoformat()
    
    # 🚨 ----------------------------------------------------------------------
    # PRUEBA DE ESTRES INYECTADA MANUALMENTE: FORZAR FALLA DESTRUCTIVA EN SENSOR_01
    # ----------------------------------------------------------------------
    if sensor_id == "SENSOR_01":
        return {
            "timestamp": timestamp,
            "sensor_id": sensor_id,
            "temperatura": 135.8,                    # Calor extremo (Motor fundiéndose)
            "voltaje": 45.2,                         # Caída crítica de tensión
            "vibracion": 0.9850,                     # Vibración destructiva fuera de rango habitual
            "estado_alerta": "SIGNAL_LOSS"           # Etiqueta objetivo para auditar el Recall de la IA
        }
    # ----------------------------------------------------------------------
    
    # 1. Inyección de ANOMALÍA ESTADÍSTICA (Para el resto de los sensores)
    if probabilidad < 0.02:
        return {
            "timestamp": timestamp,
            "sensor_id": sensor_id,
            "temperatura": round(config["temp_base"] * random.uniform(2.5, 4.0), 2),
            "voltaje": round(config["volt_base"] * random.uniform(0.1, 0.4), 2),
            "vibracion": round(config["vib_base"] * random.uniform(5.0, 10.0), 4),
            "estado_alerta": "CRITICAL_ERROR"
        }
    
    # 2. Inyección de VALORES NULOS
    elif probabilidad < 0.05:
        return {
            "timestamp": timestamp,
            "sensor_id": sensor_id,
            "temperatura": None,
            "voltaje": round(config["volt_base"] + random.uniform(-5, 5), 2),
            "vibracion": None,
            "estado_alerta": "SIGNAL_LOSS"
        }
    
    # 3. Lectura NORMAL (Para el resto de los sensores)
    else:
        return {
            "timestamp": timestamp,
            "sensor_id": sensor_id,
            "temperatura": round(config["temp_base"] + random.uniform(-1.5, 1.5), 2),
            "voltaje": round(config["volt_base"] + random.uniform(-3.0, 3.0), 2),
            "vibracion": round(config["vib_base"] + random.uniform(-0.005, 0.005), 4),
            "estado_alerta": "OK"
        }
def simular_ingesta_masiva(ciclos=10, registros_por_ciclo=100):
    """Genera ráfagas de datos simulando un lote (batch) de mensajes JSON"""
    
    # 🌟 AÑADE ESTA LÍNEA AQUÍ PARA CREAR LA CARPETA SI NO EXISTE
    os.makedirs(BRONZE_PATH, exist_ok=True)
    
    print(f"🚀 Iniciando simulación de ingesta en capa BRONZE...")
    
    for ciclo in range(ciclos):
        lote_datos = []
        for _ in range(registros_por_ciclo):
            sensor_id = random.choice(list(SENSORES_CONFIG.keys()))
            lectura = generar_lectura_sensor(sensor_id, SENSORES_CONFIG[sensor_id])
            lote_datos.append(lectura)
            
            # Inyección de DUPLICADOS - 5% de probabilidad
            if random.random() < 0.05:
                lote_datos.append(lectura)
        
        # Guardar el lote en un archivo JSON único en la capa Bronze
        filename = f"telemetry_batch_{int(time.time())}_{ciclo}.json"
        file_path = os.path.join(BRONZE_PATH, filename)
        
        with open(file_path, 'w') as f:
            json.dump(lote_datos, f, indent=4)
            
        print(f"📦 Guardado lote {ciclo+1}/{ciclos} con {len(lote_datos)} registros en Bronze.")
        time.sleep(1)

In [36]:
# Ejecutar una simulación de prueba para generar unos miles de datos
if __name__ == "__main__":
    # Generará 10 archivos masivos en la carpeta bronze
    simular_ingesta_masiva(ciclos=10, registros_por_ciclo=500)

🚀 Iniciando simulación de ingesta en capa BRONZE...
📦 Guardado lote 1/10 con 522 registros en Bronze.
📦 Guardado lote 2/10 con 514 registros en Bronze.
📦 Guardado lote 3/10 con 522 registros en Bronze.
📦 Guardado lote 4/10 con 523 registros en Bronze.
📦 Guardado lote 5/10 con 525 registros en Bronze.
📦 Guardado lote 6/10 con 530 registros en Bronze.
📦 Guardado lote 7/10 con 527 registros en Bronze.
📦 Guardado lote 8/10 con 528 registros en Bronze.
📦 Guardado lote 9/10 con 535 registros en Bronze.
📦 Guardado lote 10/10 con 524 registros en Bronze.
